# 🔬 DenseNet-169 for Skin Lesion Classification — ISIC 2024 (SLICE-3D)

> **Deep Learning Assignment — 6th Semester, May 2026**  
> **Instructor:** Dr. Baijnath Kaushik  
> **Team:** Sourbh Sharma (23BCS089) · Manish Kumar (23BCS050) · Ayush Patel (23BCS022)

---
This notebook trains and evaluates a **DenseNet-169** CNN on the ISIC 2024 SLICE-3D dataset
for binary skin-lesion classification (Benign vs Malignant).
The pipeline covers: data download → preprocessing → training → evaluation → visualisation.


## 1. Environment Setup

In [ ]:
# Install / upgrade compatible PyTorch (cu121) and dependencies
!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 \
    --index-url https://download.pytorch.org/whl/cu121 -q
!pip install pandas scikit-learn pillow tqdm matplotlib seaborn kaggle -q
print('✅ Dependencies installed')

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

## 2. Download ISIC 2024 Dataset from Kaggle

In [ ]:
import os

# ── Kaggle API token ──────────────────────────────────────────────────────
# Option A (Kaggle notebook): token is pre-configured
# Option B (local/Colab): upload kaggle.json and run the two lines below
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c isic-2024-challenge
!mkdir -p data/isic2024
!unzip -oq isic-2024-challenge.zip -d data/isic2024
!du -sh data/isic2024

In [ ]:
!find data/isic2024 -name 'train-metadata.csv'
import pandas as pd
df_preview = pd.read_csv('data/isic2024/train-metadata.csv', low_memory=False)
print(f'Shape : {df_preview.shape}')
print(f"Malignant : {df_preview['target'].sum():,}")
print(f"Benign    : {(df_preview['target']==0).sum():,}")
df_preview[['isic_id','target']].head()

## 3. Configuration (`config.py`)

In [ ]:
%%writefile config.py
"""
config.py
---------
Central configuration for DenseNet-169 on ISIC 2024 (SLICE-3D).
"""

import torch

CFG = {
    "data_dir":      "./data/isic2024",
    "img_dir":       "./data/isic2024/train-image/image/",
    "metadata_csv":  "./data/isic2024/train-metadata.csv",
    "output_dir":    "./outputs",
    "weights_path":  "./outputs/best_densenet169.pth",
    "model_name":    "densenet169",
    "num_classes":   1,
    "pretrained":    True,
    "dropout":       0.4,
    "img_size":      224,
    "seed":          42,
    "epochs":        30,
    "batch_size":    32,
    "num_workers":   4,
    "pin_memory":    True,
    "lr":            1e-4,
    "weight_decay":  1e-4,
    "scheduler":     "cosine",
    "pos_weight":    24.0,
    "threshold":     0.5,
    "train_frac":    0.70,
    "val_frac":      0.15,
    "test_frac":     0.15,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}


## 4. Dataset & DataLoaders (`dataset.py`)

In [ ]:
%%writefile dataset.py
"""
dataset.py — PyTorch Dataset and DataLoader factory for ISIC 2024.
"""

import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

from config import CFG


def get_transforms(split):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    size = CFG['img_size']
    if split == 'train':
        return T.Compose([
            T.Resize((size, size)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomRotation(degrees=15),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.0),
            T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            T.ToTensor(),
            T.Normalize(mean, std),
        ])
    else:
        return T.Compose([
            T.Resize((size, size)),
            T.ToTensor(),
            T.Normalize(mean, std),
        ])


class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{row['isic_id']}.jpg")
        image    = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(int(row['target']), dtype=torch.float32)
        return image, label


def stratified_split(metadata_csv, seed=CFG['seed']):
    df = pd.read_csv(metadata_csv, low_memory=False)
    train_df, temp_df = train_test_split(
        df, test_size=0.30, stratify=df['target'], random_state=seed)
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df['target'], random_state=seed)
    print(f"Train : {len(train_df):>7,}  (pos={train_df['target'].sum():,})")
    print(f"Val   : {len(val_df):>7,}  (pos={val_df['target'].sum():,})")
    print(f"Test  : {len(test_df):>7,}  (pos={test_df['target'].sum():,})")
    return train_df, val_df, test_df


def make_weighted_sampler(df):
    counts  = df['target'].value_counts()
    class_w = {c: 1.0 / counts[c] for c in counts.index}
    weights = df['target'].map(class_w).values
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(weights),
        num_samples=len(weights),
        replacement=True,
    )


def get_loaders(metadata_csv=CFG['metadata_csv'], img_dir=CFG['img_dir'],
                batch_size=CFG['batch_size'], num_workers=CFG['num_workers'],
                use_sampler=True):
    train_df, val_df, test_df = stratified_split(metadata_csv)
    train_ds = ISICDataset(train_df, img_dir, get_transforms('train'))
    val_ds   = ISICDataset(val_df,   img_dir, get_transforms('val'))
    test_ds  = ISICDataset(test_df,  img_dir, get_transforms('test'))
    sampler  = make_weighted_sampler(train_df) if use_sampler else None
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              shuffle=(sampler is None), num_workers=num_workers,
                              pin_memory=CFG['pin_memory'])
    val_loader  = DataLoader(val_ds,  batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=CFG['pin_memory'])
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=CFG['pin_memory'])
    return train_loader, val_loader, test_loader


## 5. Model Architecture (`model.py`)

In [ ]:
%%writefile model.py
"""
model.py — DenseNet-169 for binary skin-lesion classification.
"""

import torch
import torch.nn as nn
import torchvision.models as models
from config import CFG


class DenseNet169Classifier(nn.Module):
    def __init__(self, pretrained=True, dropout=0.4):
        super().__init__()
        weights        = 'IMAGENET1K_V1' if pretrained else None
        backbone       = models.densenet169(weights=weights)
        self.features  = backbone.features
        self.pool      = nn.AdaptiveAvgPool2d(1)
        in_features    = backbone.classifier.in_features  # 1664
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(512, 1),
        )

    def forward(self, x):
        feat   = self.features(x)
        feat   = self.pool(feat)
        feat   = feat.view(feat.size(0), -1)
        logits = self.classifier(feat)
        return logits.squeeze(1)


def build_model(pretrained=CFG['pretrained'], dropout=CFG['dropout']):
    model     = DenseNet169Classifier(pretrained=pretrained, dropout=dropout)
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'DenseNet-169 | Total params: {total:,} | Trainable: {trainable:,}')
    return model


def freeze_backbone(model):
    for param in model.features.parameters():
        param.requires_grad = False
    print('Backbone frozen.')


def unfreeze_backbone(model):
    for param in model.features.parameters():
        param.requires_grad = True
    print('Backbone unfrozen.')


## 6. Training Loop (`train.py`)

In [ ]:
%%writefile train.py
"""
train.py — Training loop for DenseNet-169 on ISIC 2024.
"""

import os, json, random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, roc_auc_score

from config import CFG
from dataset import get_loaders
from model import build_model


def set_seed(seed=CFG['seed']):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


def run_epoch(model, loader, criterion, optimizer=None, device='cpu'):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for imgs, lbls in tqdm(loader, leave=False, desc='Train' if is_train else 'Eval '):
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            loss   = criterion(logits, lbls)
            if is_train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    avg_loss   = total_loss / len(all_labels)
    acc        = accuracy_score(all_labels, (all_probs > CFG['threshold']).astype(int))
    return avg_loss, acc, all_probs, all_labels


def train():
    set_seed()
    os.makedirs(CFG['output_dir'], exist_ok=True)
    device = CFG['device']
    print(f'Device: {device}\n')

    train_loader, val_loader, _ = get_loaders(
        metadata_csv=CFG['metadata_csv'], img_dir=CFG['img_dir'],
        batch_size=CFG['batch_size'], num_workers=CFG['num_workers'], use_sampler=True)

    model     = build_model().to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([CFG['pos_weight']], device=device))
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])

    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'val_auc':[]}
    best_auc, best_epoch = 0.0, 0

    print('=' * 70)
    print(f"{'Ep':>3}  {'TrLoss':>8}  {'VlLoss':>8}  {'TrAcc':>7}  {'VlAcc':>7}  {'VlAUC':>7}  {'LR':>10}")
    print('=' * 70)

    for epoch in range(1, CFG['epochs'] + 1):
        tr_loss, tr_acc, _, _    = run_epoch(model, train_loader, criterion, optimizer, device)
        vl_loss, vl_acc, vp, vl = run_epoch(model, val_loader, criterion, None, device)
        vl_auc     = roc_auc_score(vl, vp) if len(np.unique(vl)) > 1 else 0.0
        current_lr = scheduler.get_last_lr()[0]
        scheduler.step()

        for k, v in zip(['train_loss','val_loss','train_acc','val_acc','val_auc'],
                        [tr_loss, vl_loss, tr_acc, vl_acc, vl_auc]):
            history[k].append(v)

        flag = ''
        if vl_auc > best_auc:
            best_auc, best_epoch = vl_auc, epoch
            torch.save(model.state_dict(), CFG['weights_path'])
            flag = '  ✓ saved'

        print(f"{epoch:>3}  {tr_loss:>8.4f}  {vl_loss:>8.4f}  "
              f"{tr_acc:>7.4f}  {vl_acc:>7.4f}  {vl_auc:>7.4f}  {current_lr:>10.2e}{flag}")

    print('=' * 70)
    print(f'Best AUC {best_auc:.4f} at epoch {best_epoch}')

    hist_path = os.path.join(CFG['output_dir'], 'history.json')
    with open(hist_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f'History saved → {hist_path}')
    return history


if __name__ == '__main__':
    train()


## 7. Evaluation (`evaluate.py`)

In [ ]:
%%writefile evaluate.py
"""
evaluate.py — Load best checkpoint and evaluate on test set.
"""

import os, json
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
)

from config import CFG
from dataset import get_loaders
from model import build_model


def run_inference(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc='Testing'):
            probs = torch.sigmoid(model(imgs.to(device))).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(lbls.numpy())
    return np.array(all_probs), np.array(all_labels)


def find_best_threshold(val_probs, val_labels):
    fpr, tpr, thresholds = roc_curve(val_labels, val_probs)
    j_scores = tpr - fpr
    return float(thresholds[np.argmax(j_scores)])


def plot_confusion_matrix(cm, save_path):
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Benign','Malignant'], yticklabels=['Benign','Malignant'])
    ax.set_title('Confusion Matrix — DenseNet-169 on ISIC 2024', fontsize=13)
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Saved → {save_path}')


def plot_roc_curve(labels, probs, auc, save_path):
    fpr, tpr, _ = roc_curve(labels, probs)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, lw=2, color='#1f77b4', label=f'DenseNet-169  (AUC = {auc:.4f})')
    ax.plot([0,1],[0,1],'--',color='gray',lw=1,label='Random')
    ax.fill_between(fpr, tpr, alpha=0.08, color='#1f77b4')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve — DenseNet-169 on ISIC 2024', fontsize=13)
    ax.legend(loc='lower right'); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Saved → {save_path}')


def plot_loss_acc_curves(history_path, save_path):
    with open(history_path) as f:
        hist = json.load(f)
    epochs = range(1, len(hist['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, hist['train_loss'], label='Train')
    axes[0].plot(epochs, hist['val_loss'],   label='Val')
    axes[0].set_title('Loss Curve'); axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('BCE Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(epochs, hist['train_acc'], label='Train Acc')
    axes[1].plot(epochs, hist['val_acc'],   label='Val Acc')
    if 'val_auc' in hist:
        axes[1].plot(epochs, hist['val_auc'], label='Val AUC', linestyle='--')
    axes[1].set_title('Accuracy & AUC'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.suptitle('DenseNet-169 Training History — ISIC 2024', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Saved → {save_path}')


def evaluate():
    os.makedirs(CFG['output_dir'], exist_ok=True)
    device = CFG['device']
    _, val_loader, test_loader = get_loaders(
        metadata_csv=CFG['metadata_csv'], img_dir=CFG['img_dir'],
        batch_size=CFG['batch_size'], num_workers=CFG['num_workers'], use_sampler=False)
    model = build_model(pretrained=False).to(device)
    model.load_state_dict(torch.load(CFG['weights_path'], map_location=device))
    print(f'Loaded weights from {CFG["weights_path"]}')

    val_probs, val_labels   = run_inference(model, val_loader, device)
    threshold               = find_best_threshold(val_probs, val_labels)
    print(f'Optimal threshold (Youden J): {threshold:.6f}')

    test_probs, test_labels = run_inference(model, test_loader, device)
    test_preds = (test_probs > threshold).astype(int)

    acc  = accuracy_score(test_labels, test_preds)
    prec = precision_score(test_labels, test_preds, zero_division=0)
    rec  = recall_score(test_labels, test_preds, zero_division=0)
    f1   = f1_score(test_labels, test_preds, zero_division=0)
    auc  = roc_auc_score(test_labels, test_probs)
    cm   = confusion_matrix(test_labels, test_preds)

    results = {'threshold': float(threshold), 'accuracy': float(acc),
               'precision': float(prec), 'recall': float(rec),
               'f1_score': float(f1), 'auc_roc': float(auc),
               'confusion_matrix': cm.tolist()}

    print('\n' + '='*50)
    print('        TEST SET RESULTS — DenseNet-169')
    print('='*50)
    for k, v in results.items():
        if k != 'confusion_matrix':
            print(f'  {k:<12}: {v:.4f}')
    print('='*50)
    print('\nClassification Report:')
    print(classification_report(test_labels, test_preds,
                                target_names=['Benign','Malignant'], zero_division=0))

    res_path = os.path.join(CFG['output_dir'], 'test_results.json')
    with open(res_path, 'w') as f:
        json.dump(results, f, indent=2)

    plot_confusion_matrix(cm, os.path.join(CFG['output_dir'], 'confusion_matrix.png'))
    plot_roc_curve(test_labels, test_probs, auc, os.path.join(CFG['output_dir'], 'roc_curve.png'))
    hist_path = os.path.join(CFG['output_dir'], 'history.json')
    if os.path.exists(hist_path):
        plot_loss_acc_curves(hist_path, os.path.join(CFG['output_dir'], 'loss_acc_curves.png'))
    return results


if __name__ == '__main__':
    evaluate()


## 8. Run Training

> Trains for **30 epochs** with CosineAnnealingLR.  
> Best checkpoint saved to `outputs/best_densenet169.pth` (tracked by val AUC-ROC).


In [ ]:
import os
os.makedirs('outputs', exist_ok=True)

from train import train
history = train()

## 9. Run Evaluation

Loads the best checkpoint, finds the optimal decision threshold via **Youden's J statistic** on the validation set, then reports all metrics on the held-out test set.


In [ ]:
from evaluate import evaluate
results = evaluate()

## 10. Visualise Results

In [ ]:
import json
import matplotlib.pyplot as plt
%matplotlib inline

with open('outputs/history.json') as f:
    hist = json.load(f)

epochs = range(1, len(hist['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, hist['train_loss'], label='Train Loss')
axes[0].plot(epochs, hist['val_loss'],   label='Val Loss')
axes[0].set_title('Loss Curve'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, hist['train_acc'], label='Train Acc')
axes[1].plot(epochs, hist['val_acc'],   label='Val Acc')
axes[1].plot(epochs, hist['val_auc'],   label='Val AUC', linestyle='--')
axes[1].set_title('Accuracy & AUC Curve'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('DenseNet-169 Training History — ISIC 2024', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import Image, display

print('=== Confusion Matrix ===')
display(Image('outputs/confusion_matrix.png'))
print('\n=== ROC Curve ===')
display(Image('outputs/roc_curve.png'))

## 11. Final Metrics Summary

In [ ]:
import json
with open('outputs/test_results.json') as f:
    res = json.load(f)

print('╔══════════════════════════════════════════╗')
print('║   TEST SET RESULTS — DenseNet-169        ║')
print('╠══════════════════════════════════════════╣')
print(f"║  AUC-ROC   : {res['auc_roc']:.4f}                      ║")
print(f"║  Accuracy  : {res['accuracy']:.4f}                      ║")
print(f"║  Precision : {res['precision']:.4f}                      ║")
print(f"║  Recall    : {res['recall']:.4f}                      ║")
print(f"║  F1-Score  : {res['f1_score']:.4f}                      ║")
print(f"║  Threshold : {res['threshold']:.2e}                ║")
print('╠══════════════════════════════════════════╣')
cm = res['confusion_matrix']
print(f"║  TN={cm[0][0]:>6}  FP={cm[0][1]:>6}                   ║")
print(f"║  FN={cm[1][0]:>6}  TP={cm[1][1]:>6}                   ║")
print('╚══════════════════════════════════════════╝')